In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Step 1 — Load
df = pd.read_parquet('/content/drive/MyDrive/airbnb_lab/airbnb_clean.parquet')
print(df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(21157, 21)


,id,host_id,neighbourhood,neighbourhood_group_cleansed,room_type,latitude,longitude,accommodates,bedrooms,bathrooms,...,availability_365,number_of_reviews,reviews_per_year,review_scores_rating,host_age_days,distance_to_times_square_km,mean_review_length,description,price,log_price
0,2595,2845,Neighborhood highlights,Manhattan,Entire home/apt,40.75356,-73.98559,1,0.0,1.0,...,299,49,3.152097,4.68,6112,0.493764,370.191489,"Beautiful, spacious skylit studio in the heart...",302.0,5.713733
1,5136,7378,<NA>,Brooklyn,Entire home/apt,40.66265,-73.99454,4,2.0,1.5,...,67,4,0.350036,4.75,5965,10.629781,0.000000,"We welcome you to stay in our lovely 2 br, 130...",215.0,5.375278
2,6848,15991,<NA>,Brooklyn,Entire home/apt,40.70935,-73.95342,3,2.0,1.0,...,206,195,12.158353,4.58,5873,6.047341,338.055838,Comfortable studio apartment with super comfor...,96.0,4.574711
3,6872,16104,Neighborhood highlights,Manhattan,Private room,40.80107,-73.94255,1,1.0,1.0,...,63,1,0.333333,5.00,5872,6.001197,279.000000,This charming distancing-friendly month-to-mon...,91.0,4.521789
4,6990,16800,Neighborhood highlights,Manhattan,Private room,40.78778,-73.94759,1,2.0,1.0,...,241,253,16.206564,4.88,5867,4.599598,430.776892,Beautiful peaceful healthy home,59.0,4.094345


In [2]:
# Step 2 — Define X and y  (columns are listed per scenario below)

feature_columns = [
    'accommodates', 'bedrooms', 'bathrooms', 'number_of_reviews',
    'reviews_per_year', 'host_age_days', 'distance_to_times_square_km'
]

X = df[feature_columns]
y = df['log_price']


In [3]:
# Step 3 — Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Step 4 — Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)


In [5]:
# Step 5 — Train Model 1 and Model 2  (models listed per scenario below)
model1 = LinearRegression()
model2 = RandomForestRegressor(n_estimators=100, random_state=42)

model1.fit(X_train, y_train)
model2.fit(X_train, y_train)

y_pred_1 = model1.predict(X_test)
y_pred_2 = model2.predict(X_test)

In [6]:
# Step 6 — Evaluate
mae_1  = mean_absolute_error(y_test, y_pred_1)
rmse_1 = np.sqrt(mean_squared_error(y_test, y_pred_1))
r2_1   = r2_score(y_test, y_pred_1)

mae_2  = mean_absolute_error(y_test, y_pred_2)
rmse_2 = np.sqrt(mean_squared_error(y_test, y_pred_2))
r2_2   = r2_score(y_test, y_pred_2)

In [7]:
metrics_summary = pd.DataFrame({
    'Metric': ['Mean Absolute Error (MAE)', 'Root Mean Squared Error (RMSE)', 'R² Score'],
    'Linear Regression': [mae_1, rmse_1, r2_1],
    'Random Forest': [mae_2, rmse_2, r2_2]
})

metrics_summary = metrics_summary.round(4)
metrics_summary

,Metric,Linear Regression,Random Forest
0,Mean Absolute Error (MAE),0.4581,0.2951
1,Root Mean Squared Error (RMSE),0.5697,0.4080
2,R² Score,0.4151,0.7000


# Step 7 — Answer the 3 questions in a markdown cell

1. Higher $R^2$ and MeaningWinner: Random Forest ($0.6997$ vs. $0.4151$).In plain words: The Random Forest model successfully explains 70% of the factors that cause NYC Airbnb prices to change. Linear Regression only explains 41.5%, leaving the remaining 58.5% completely unaccounted for.
2. Log MAE = 0.3 in Real DollarsThe impact: Because the model predicts a log scale, an error of 0.3 translates to a percentage error in the real world ($e^{0.3} \approx 1.35$).In plain dollars: Your price hints are off by roughly 35% on average (e.g., the model will predict around \$135 for a room that actually costs \$100).
3. Did Random Forest Beat Linear Regression? Yes, it dominated. Random Forest captured 28.5% more of the total price variance ($R^2$ jumped from ~0.42 to ~0.70) and cut down the average prediction error by more than a third (MAE dropped from 0.458 to 0.295).